In [2]:
import pandas as pd
import requests
from xml.etree import ElementTree
import time
from tqdm import tqdm

import os

In [3]:
def fetch_pmids_by_journal(journal_abbrev, start_year=2001, end_year=2024):
    """
    Fetch PMIDs for a given PubMed journal abbreviation.
    Example journal_abbrev: 'J Biomed Inform'
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    # params = {
    #     "db": "pubmed",
    #     "term": f"{journal_abbrev}[Journal] AND ({start_year}:{end_year}[DP])",
    #     "retmode": "xml",
    #     "retmax": 100000
    # }
    params = {
        "db": "pubmed",
        "term": f"{journal_abbrev}[Journal]",
        "mindate": start_year,
        "maxdate": end_year,
        "datetype": "pdat",   # publication date
        "retmode": "xml",
        "retmax": 100000
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    root = ElementTree.fromstring(response.content)

    return [id_elem.text for id_elem in root.findall(".//Id")]



def fetch_pubmed_records(pmids, batch_size=200):
    """
    Fetch metadata for a list of PMIDs in batches.
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    all_records = []

    for i in tqdm(range(0, len(pmids), batch_size)):
        batch = pmids[i:i+batch_size]
        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "xml"
        }
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        root = ElementTree.fromstring(response.content)

        for article in root.findall(".//PubmedArticle"):
            record = {
                "pmid": article.findtext(".//PMID"),
                "title": article.findtext(".//ArticleTitle"),
                "abstract": article.findtext(".//Abstract/AbstractText"),
                "year": article.findtext(".//JournalIssue/PubDate/Year"),
                "mesh_terms": [mh.findtext("DescriptorName") for mh in article.findall(".//MeshHeading")],
                "keywords": [kw.text for kw in article.findall(".//KeywordList/Keyword") if kw.text]
            }
            all_records.append(record)

        time.sleep(0.34)  # <= 3 requests/second (NCBI policy)

    return all_records





In [ ]:
start_year = 2001
end_year = 2025 
pmids = fetch_pmids_by_journal("J Biomed Inform", start_year, end_year)
print(f"Found {len(pmids)} JBI articles ({start_year}–{end_year})")

records = fetch_pubmed_records(pmids, batch_size=200)
print(f"Retrieved {len(records)} records")
print(records[0])  # peek at first record


In [ ]:
df = pd.DataFrame(records)
df["mesh_terms"] = df["mesh_terms"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df["keywords"] = df["keywords"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df = df.loc[(df['year']<=end_year) & (df['year']>=start_year)]
print("number of articles:", len(df))

folder_path = "data/raw"
os.makedirs(folder_path, exist_ok=True)
df.to_csv(os.path.join(folder_path, f"jbi_pubmed_{start_year}_{end_year}.csv"), index=False)

In [ ]:
n = 0
for record in df['mesh_terms']:
    if record!="":
        n += 1
print("number of records with MeSH terms:", n)

In [ ]:
folder_path = "data/raw"
df = pd.read_csv(os.path.join(folder_path, f"jbi_pubmed_2001_2025.csv"))

df = df.loc[(df['keywords']!="")&df['keywords'].notna()]
print(df.shape, df.year.min(), df.year.max())

df.to_csv(os.path.join(folder_path, f"jbi_pubmed_{df.year.min()}_{df.year.max()}_w_keywords.csv"), index=False)